# Checkpointers and BaseStore

LangGraph separates **two persistence layers** for agent memory:

| Layer | API | Scope | Typical data |
|-------|-----|-------|--------------|
| **Checkpointer** | `compile(checkpointer=...)` | **Thread** (`thread_id`) | Messages, turn scratch, HITL resume |
| **Store** | `compile(store=...)` + `BaseStore` | **User / namespace** | Preferences, facts across new chats |

```
  thread_id=chat-001          user_id=alice
       │                           │
       ▼                           ▼
  Checkpointer               BaseStore (InMemoryStore)
  ─ messages                 ─ namespace: ("shopco", "alice")
  ─ order_id (scratch)       ─ key: "preferences"
  ─ turn_count               ─ value: {contact: email, tier: pro}
```

Use **both** in production ShopCo-style agents: checkpointers for the active ticket thread; `BaseStore` for long-term customer context that survives **new** threads.

This notebook implements **ShopCo Support** with `MemorySaver` + `InMemoryStore`, demonstrates cross-thread recall, and maps the pattern to production Postgres backends.

In [ ]:
"""
ShopCo Support — checkpointers + BaseStore lab.
"""

import json
import os
import re
import uuid
from typing import Annotated, Any

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore
from typing_extensions import TypedDict

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


POLICY_SNIPPETS = {
    "returns": "Returns within 30 days. [pol-returns-01]",
    "shipping": "Free shipping over $50. [pol-shipping-02]",
}

ORDERS_DB = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped"},
    "ORD-1002": {"order_id": "ORD-1002", "status": "processing"},
}

print("Environment ready.")

---

## 1. Checkpointer — Thread-Scoped State

A **checkpointer** saves graph state after each super-step. Keyed by `thread_id` in `config["configurable"]`.

- `MemorySaver` — in-process, dev/test
- `PostgresSaver` / `SqliteSaver` — production durability

Thread state dies when you start a **new** `thread_id` — unless you copy data to the store.

---

## 2. BaseStore — Namespace-Scoped Long-Term Memory

`BaseStore` organizes data as:

- **namespace** — tuple path, e.g. `("shopco", "alice")`
- **key** — unique id within namespace, e.g. `"profile"`
- **value** — JSON-serializable dict

Core operations: `put`, `get`, `search`, `delete`.

In [ ]:
ltm_store = InMemoryStore()

namespace_alice = ("shopco", "alice")
ltm_store.put(namespace_alice, "profile", {
    "customer_name": "Alice",
    "contact_channel": "email",
    "tier": "pro",
})
ltm_store.put(namespace_alice, "mem-001", {
    "fact": "Prefers concise answers",
    "context": "stated in prior chat",
})

profile = ltm_store.get(namespace_alice, "profile")
print("Profile:", profile.value if profile else None)

memories = ltm_store.search(namespace_alice)
print("Items in namespace:", [m.key for m in memories])

---

## 3. Graph State — Thread Fields + Loaded Memory Summary

In [ ]:
class SupportState(TypedDict):
    messages: Annotated[list, add_messages]
    order_id: str
    customer_name: str
    turn_count: int
    memory_context: str


def user_namespace(config: RunnableConfig) -> tuple[str, ...]:
    user_id = config.get("configurable", {}).get("user_id", "anonymous")
    return ("shopco", user_id)


def extract_thread_entities(state: SupportState) -> dict[str, Any]:
    last = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    if not last:
        return {"turn_count": state.get("turn_count", 0)}
    text = str(last.content)
    updates: dict[str, Any] = {"turn_count": state.get("turn_count", 0) + 1}
    m = re.search(r"ORD-[0-9]{4}", text.upper())
    if m:
        updates["order_id"] = m.group(0)
    name_m = re.search(r"my name is ([A-Za-z]+)", text, re.I)
    if name_m:
        updates["customer_name"] = name_m.group(0).split()[-1]
    return updates


print("SupportState ready.")

---

## 4. Store Nodes — Load and Persist Long-Term Memory

In [ ]:
def load_long_term_memory(state: SupportState, config: RunnableConfig, *, store: BaseStore) -> dict[str, Any]:
    """Read user namespace from BaseStore into thread state for this turn."""
    ns = user_namespace(config)
    items = store.search(ns)
    if not items:
        return {"memory_context": "(no long-term memory)"}
    lines = [f"{item.key}: {item.value}" for item in items]
    updates: dict[str, Any] = {"memory_context": "\n".join(lines)}
    profile_item = store.get(ns, "profile")
    if profile_item and isinstance(profile_item.value, dict):
        if profile_item.value.get("customer_name") and not state.get("customer_name"):
            updates["customer_name"] = profile_item.value["customer_name"]
        if profile_item.value.get("last_order_id") and not state.get("order_id"):
            updates["order_id"] = profile_item.value["last_order_id"]
    return updates


def persist_long_term_memory(state: SupportState, config: RunnableConfig, *, store: BaseStore) -> dict[str, Any]:
    """Write durable user facts from thread scratch to BaseStore."""
    ns = user_namespace(config)
    profile = {
        "customer_name": state.get("customer_name") or "",
        "last_order_id": state.get("order_id") or "",
    }
    if profile["customer_name"] or profile["last_order_id"]:
        store.put(ns, "profile", profile)

    last = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    if last and "prefer" in str(last.content).lower():
        store.put(ns, f"mem-{uuid.uuid4().hex[:8]}", {
            "fact": str(last.content),
            "type": "preference",
        })
    return {}


print("Store nodes defined.")

---

## 5. Respond Node — Uses Thread + Store Context

In [ ]:
def respond_node(state: SupportState, config: RunnableConfig) -> dict[str, Any]:
    last = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    q = str(last.content).lower() if last else ""
    name = state.get("customer_name") or "there"
    order_id = state.get("order_id", "")
    mem = state.get("memory_context", "")

    if "return" in q:
        body = POLICY_SNIPPETS["returns"]
    elif "shipping" in q:
        body = POLICY_SNIPPETS["shipping"]
    elif "status" in q or ("it" in q and order_id):
        row = ORDERS_DB.get(order_id, {})
        body = f"Order {order_id or '?'} status: {row.get('status', 'unknown')}."
    elif "remember" in q and "prefer" in q:
        body = "Got it — I'll remember that preference for future chats."
    else:
        body = "How can I help with your ShopCo order or policy?"

    if mem and mem != "(no long-term memory)" and "prefer" not in q:
        body += f" (I also have your saved profile/preferences on file.)"

    return {"messages": [AIMessage(content=f"Hi {name}, {body}")]}


def live_respond_node(state: SupportState, config: RunnableConfig) -> dict[str, Any]:
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    system = SystemMessage(content=(
        f"ShopCo support. Thread order={state.get('order_id')}. "
        f"Long-term memory:\n{state.get('memory_context', '')}"
    ))
    resp = llm.invoke([system] + state["messages"])
    return {"messages": [resp]}


def respond(state: SupportState, config: RunnableConfig) -> dict[str, Any]:
    if USE_LIVE_LLM:
        return live_respond_node(state, config)
    return respond_node(state, config)

---

## 6. Compile with Checkpointer **and** Store

In [ ]:
def build_graph(store: BaseStore, checkpointer: MemorySaver | None = None):
    builder = StateGraph(SupportState)
    builder.add_node("load_memory", load_long_term_memory)
    builder.add_node("extract", extract_thread_entities)
    builder.add_node("respond", respond)
    builder.add_node("persist_memory", persist_long_term_memory)

    builder.add_edge(START, "load_memory")
    builder.add_edge("load_memory", "extract")
    builder.add_edge("extract", "respond")
    builder.add_edge("respond", "persist_memory")
    builder.add_edge("persist_memory", END)

    return builder.compile(checkpointer=checkpointer, store=store)


SHARED_STORE = InMemoryStore()
SHARED_CHECKPOINTER = MemorySaver()
GRAPH = build_graph(store=SHARED_STORE, checkpointer=SHARED_CHECKPOINTER)

print("Graph compiled with MemorySaver + InMemoryStore")

---

## 7. Invoke Helper — thread_id + user_id

In [ ]:
def send_message(graph, user_id: str, thread_id: str, text: str) -> SupportState:
    config: RunnableConfig = {
        "configurable": {"thread_id": thread_id, "user_id": user_id},
    }
    return graph.invoke({"messages": [HumanMessage(content=text)]}, config=config)


def last_ai(state: SupportState) -> str:
    for m in reversed(state["messages"]):
        if isinstance(m, AIMessage):
            return str(m.content)
    return ""

---

## 8. Thread Memory — Multi-Turn on One thread_id

In [ ]:
uid, tid = "alice", "ticket-100"

s1 = send_message(GRAPH, uid, tid, "My name is Alice. Order ORD-1001.")
print("Turn 1:", last_ai(s1))
print("  thread order_id:", s1["order_id"])

s2 = send_message(GRAPH, uid, tid, "Can I return it?")
print("\nTurn 2:", last_ai(s2))
print("  messages in checkpoint:", len(s2["messages"]))

---

## 9. Long-Term Memory — New Thread, Same user_id

Customer opens a **new ticket** (`thread_id`) days later. Checkpointer is empty, but **BaseStore** recalls profile.

In [ ]:
new_thread = "ticket-999"  # brand-new conversation
s3 = send_message(GRAPH, uid, new_thread, "Hi, what do you know about me?")
print("New thread reply:", last_ai(s3))
print("Loaded memory context:")
print(s3["memory_context"])

profile = SHARED_STORE.get(("shopco", uid), "profile")
print("\nStore profile:", profile.value if profile else None)

---

## 10. Isolation — Different user_id, Same Store Backend

In [ ]:
send_message(GRAPH, "bob", "ticket-b1", "My name is Bob. ORD-1002.")
bob_reply = send_message(GRAPH, "bob", "ticket-b2", "What is my order status?")
alice_reply = send_message(GRAPH, "alice", "ticket-a2", "What is my order status?")

print("Bob:", last_ai(bob_reply))
print("Alice:", last_ai(alice_reply))
assert "ORD-1002" in last_ai(bob_reply)
assert "ORD-1001" in last_ai(alice_reply)
print("\nUser namespaces isolated in BaseStore")

---

## 11. Inspect Checkpointer vs Store

In [ ]:
cfg = {"configurable": {"thread_id": tid, "user_id": uid}}
checkpoint = GRAPH.get_state(cfg)

print("=== Checkpointer (thread)")
print("  turn_count:", checkpoint.values.get("turn_count"))
print("  order_id:", checkpoint.values.get("order_id"))
print("  message count:", len(checkpoint.values.get("messages", [])))

print("\n=== BaseStore (user namespace)")
for item in SHARED_STORE.search(("shopco", uid)):
    print(f"  key={item.key} value={item.value}")

---

## 12. Preference Memory Across Threads

In [ ]:
send_message(GRAPH, "carol", "t-c1", "My name is Carol. I prefer short bullet answers.")
follow = send_message(GRAPH, "carol", "t-c2", "What's the return policy?")

print("Follow-up in new thread:", last_ai(follow))
mems = SHARED_STORE.search(("shopco", "carol"))
print("Stored keys:", [m.key for m in mems])

---

## 13. Production Mapping

| Dev (this notebook) | Production |
|---------------------|------------|
| `MemorySaver` | `PostgresSaver` / `SqliteSaver` |
| `InMemoryStore` | `PostgresStore` / `RedisStore` |
| `thread_id` = ticket | Stable per support case |
| `user_id` = customer | Auth subject / CRM id |
| Namespace `("shopco", user_id)` | Tenant prefix + user |

Checkpointer handles **resume** and **HITL**. Store handles **CRM-like** memory across sessions.

---

## 14. Namespace Design Guidelines

| Pattern | Example | Use |
|---------|---------|-----|
| `(tenant, user_id)` | `("shopco", "alice")` | Multi-tenant SaaS |
| `(user_id, "profile")` | profile vs episodic split | Typed memory buckets |
| Episodic key = UUID | `mem-a1b2c3d4` | Append-only facts |
| Profile key fixed | `"profile"` | Upsert user card |

Never put one user's namespace under another user's `user_id`.

---

## 15. Checkpointer-Only vs Store-Only vs Both

| Setup | Multi-turn same chat | New chat same user | Survives process restart |
|-------|---------------------|--------------------|-------------------------|
| Checkpointer only | Yes | No | Only with DB checkpointer |
| Store only | Manual message pass | Yes | Only with DB store |
| **Both** | Yes | Yes | Yes (with DB backends) |

In [ ]:
checkpointer_only = build_graph(store=InMemoryStore(), checkpointer=MemorySaver())

send_message(checkpointer_only, "dana", "d-thread-1", "My name is Dana. ORD-1001.")
cp_only_new = send_message(checkpointer_only, "dana", "d-thread-2", "What's my order status?")

print("Checkpointer-only, new thread:", last_ai(cp_only_new))
print("Has order in answer:", "ORD-1001" in last_ai(cp_only_new))

---

## 16. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Store without `user_id` in config | Empty namespace | Pass `configurable.user_id` |
| Only checkpointer for preferences | Lost on new ticket | `persist_memory` → BaseStore |
| Huge blobs in checkpointer | Slow resume | Store episodic facts in BaseStore |
| Same namespace for all users | Leakage | `(tenant, user_id)` tuple |
| `InMemoryStore` in prod | Memory lost on deploy | Postgres store |

---

## 17. Optional Live LLM

Set `USE_LIVE_LLM = True` and re-run from the respond node cell onward.

In [ ]:
if USE_LIVE_LLM:
    g = build_graph(InMemoryStore(), MemorySaver())
    send_message(g, "live-user", "live-t1", "Remember I prefer email contact.")
    out = send_message(g, "live-user", "live-t2", "How should you contact me?")
    print(last_ai(out))
else:
    print("Offline demo complete.")
    print("Alice profile in store:", SHARED_STORE.get(("shopco", "alice"), "profile").value)

---

## 18. Quiz

1. What does a **checkpointer** persist vs what does **BaseStore** persist?
2. Why do you need both `thread_id` and `user_id` in config?
3. How does a user get remembered on a **new** support ticket?
4. What is a namespace tuple used for?
5. When should you switch from `MemorySaver` / `InMemoryStore` to Postgres?

---

## 19. Summary

- **Checkpointer** = thread memory (messages + scratch) for resume and multi-turn within one ticket.
- **BaseStore** = long-term memory keyed by namespace (user), surviving new threads.
- **Compile both**: `graph.compile(checkpointer=..., store=...)`.
- **Graph flow**: `load_memory` → extract → respond → `persist_memory`.

ShopCo Support showed Alice's profile persisting from `ticket-100` to `ticket-999` via `InMemoryStore`, while pronoun resolution still used the checkpointer within a single thread. Use database-backed implementations in production for durability and audit.